In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [13]:
import pandas as pd
ds=pd.read_csv('/kaggle/input/datasets/lakshyaupadhyaya/movie-columns-with-sentiment/movie_columns_with_sentiments.csv')

In [9]:
!pip install supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 16.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 10.2 MB/s eta 0:00:00


In [7]:
def create_embedding_text(row):
    parts = []
    
    if pd.notnull(row['title']):
        parts.append(f"Title: {row['title']}")
    
    if pd.notnull(row['description']):
        parts.append(f"Description: {row['description']}")
    
    if pd.notnull(row['directed_by']):
        parts.append(f"Directed by: {row['directed_by']}")
    
    if pd.notnull(row['starring']):
        parts.append(f"Starring: {row['starring']}")
    
    if pd.notnull(row['main_sentiment']):
        parts.append(f"Emotion: {row['main_sentiment']}")
    
    return "search_document: " + " | ".join(parts)

ds['embedding_text'] = ds.apply(create_embedding_text, axis=1)

In [8]:
ds['embedding_text']

0        search_document: Title: 0-41* | Description: 0...
1        search_document: Title: 1 Night | Description:...
2        search_document: Title: 1:54 | Description: Ti...
3        search_document: Title: 1st Sem | Description:...
4        search_document: Title: 2 Cool 2 Be 4gotten | ...
                               ...                        
31055    search_document: Title: Youth | Description: Y...
31056    search_document: Title: Zamana | Description: ...
31057    search_document: Title: Zero A. D. | Descripti...
31058    search_document: Title: Zi | Description: Zi (...
31059    search_document: Title: Zombeid | Description:...
Name: embedding_text, Length: 31060, dtype: object

In [10]:
from supabase import create_client
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
url = secrets.get_secret("SUPABASE_URL")      
key = secrets.get_secret("SUPABASE_KEY")    

supabase = create_client(url, key)

In [12]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    device="cuda",
    trust_remote_code=True
)

embeddings = model.encode(
    ds['embedding_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    device="cuda"
)

# Insert in batches of 100
batch_size = 100
for i in range(0, len(ds), batch_size):
    batch = ds.iloc[i:i+batch_size]
    batch_embeddings = embeddings[i:i+batch_size]
    
    records = [
        {
            "title": row.title,
            "description": row.description,
            "directed_by": row.directed_by,
            "produced_by": row.produced_by,
            "starring": row.starring,
            "country": row.country,
            "language": row.language,
            "release_date": str(row.release_date),
            "main_sentiment": row.main_sentiment,
            "main_sentiment_score": float(row.main_sentiment_score),
            "all_sentiments": row.all_sentiments,
            "embedding": batch_embeddings[j].tolist()
        }
        for j, row in enumerate(batch.itertuples())
    ]
    
    supabase.table('movies').insert(records).execute()
    print(f"Inserted rows {i} to {i + len(batch)}")

Batches:   0%|          | 0/330 [00:00<?, ?it/s]

Inserted rows 10001 to 10101
Inserted rows 10101 to 10201
Inserted rows 10201 to 10301
Inserted rows 10301 to 10401
Inserted rows 10401 to 10501
Inserted rows 10501 to 10601
Inserted rows 10601 to 10701
Inserted rows 10701 to 10801
Inserted rows 10801 to 10901
Inserted rows 10901 to 11001
Inserted rows 11001 to 11101
Inserted rows 11101 to 11201
Inserted rows 11201 to 11301
Inserted rows 11301 to 11401
Inserted rows 11401 to 11501
Inserted rows 11501 to 11601
Inserted rows 11601 to 11701
Inserted rows 11701 to 11801
Inserted rows 11801 to 11901
Inserted rows 11901 to 12001
Inserted rows 12001 to 12101
Inserted rows 12101 to 12201
Inserted rows 12201 to 12301
Inserted rows 12301 to 12401
Inserted rows 12401 to 12501
Inserted rows 12501 to 12601
Inserted rows 12601 to 12701
Inserted rows 12701 to 12801
Inserted rows 12801 to 12901
Inserted rows 12901 to 13001
Inserted rows 13001 to 13101
Inserted rows 13101 to 13201
Inserted rows 13201 to 13301
Inserted rows 13301 to 13401
Inserted rows 